# SHADOWCASE AI: scam text filter
**Veritas AI International High School Hackathon 2026 · Track 4.2 AI + Finance · Project 4.2.1 Scam text filter**

*This project is a prototype and is not financial advice.*

This notebook reproduces every number in our write-up:
1. Loads the **SMS Spam Collection** (5,574 messages, about 13% spam) and makes a **stratified** 80/20 split. The data has no dates, so a time-based split isn't possible.
2. Compares two **baselines**, "everything is not spam" and our app's own keyword rules, with **TF-IDF + naive Bayes** and **TF-IDF + logistic regression**.
3. Reports **precision and recall for spam** (spam is rare, so accuracy alone is misleading).
4. **Our own addition:** the dataset is over 10 years old, so we test on **24 modern UK scam and genuine messages** we wrote, and train an extra model that also learns from 40 *different* modern examples.
5. Shows **five messages the model gets wrong**, and lets you try your own message, with an explanation of which words made it suspicious.

All the code is in [`ml/train.py`](https://github.com/iamzainsiddique/shadowcase-ai/blob/main/ml/train.py). The playable app is at https://iamzainsiddique.github.io/shadowcase-ai/

In [ ]:
# Get the code and our modern UK message sets (the SMS dataset is downloaded from UCI by the script)
!git clone -q https://github.com/iamzainsiddique/shadowcase-ai.git
%cd shadowcase-ai
!pip -q install scikit-learn pandas

In [ ]:
import sys
sys.path.insert(0, "ml")
import train
train.main()

## Results table: precision and recall for spam

In [ ]:
import json, pandas as pd
r = json.load(open("ml/results.json"))
rows = []
for name, v in r["results"].items():
    rows.append({"Method": name,
                 "SMS precision": v["sms_test"]["precision"], "SMS recall": v["sms_test"]["recall"], "SMS accuracy": v["sms_test"]["accuracy"],
                 "Modern UK precision": v["modern_uk_test"]["precision"], "Modern UK recall": v["modern_uk_test"]["recall"]})
pd.DataFrame(rows).style.format({c: "{:.1%}" for c in rows[0] if c != "Method"})

Note: "everything is not spam" scores about 87% accuracy but catches nothing (recall 0%). That's why we report precision and recall.

Caution: our keyword rules were written while looking at the 24 modern UK test messages, so their modern-test score is optimistic. The trained models never saw those messages.

## Five messages the final model gets wrong

In [ ]:
for e in r["errors_sms_test"]:
    print(f"TRUE {e['true']:9s} PREDICTED {e['predicted']:9s} p(spam)={e['p_spam']}:  {e['text']}")
print("\nModern UK messages it gets wrong:")
for e in r["errors_modern_uk_test"]:
    print(f"TRUE {e['true']:9s} PREDICTED {e['predicted']:9s} p(spam)={e['p_spam']}:  {e['text']}")

## Try your own message (with an explanation)
This uses the exported model, the same one the app runs in the browser, and lists the words that pushed it towards "scam".

In [ ]:
import math, re
m = json.load(open("src/ml/scam-model.json"))
index = {t: i for i, t in enumerate(m["terms"])}

def explain(message, top=5):
    words = re.findall(r"(?u)\b\w\w+\b", message.lower())
    grams = words + [a + " " + b for a, b in zip(words, words[1:])]
    counts = {}
    for g in grams:
        if g in index: counts[g] = counts.get(g, 0) + 1
    w = {g: (1 + math.log(c)) * m["idf"][index[g]] for g, c in counts.items()}
    norm = math.sqrt(sum(v * v for v in w.values())) or 1
    contrib = {g: v / norm * m["coef"][index[g]] for g, v in w.items()}
    p = 1 / (1 + math.exp(-(sum(contrib.values()) + m["intercept"])))
    print(f"p(scam) = {p:.2f} -> {'FLAG: think twice' if p >= m['threshold'] else 'no strong scam signals (not proof it is safe)'}")
    for g, c in sorted(contrib.items(), key=lambda x: -x[1])[:top]:
        if c > 0: print(f"   '{g}' pushed it towards scam (+{c:.2f})")

explain("Your parcel is on hold. Pay the £1.99 redelivery fee within 24 hours: parcel-fee-help.info")